# WRI/GFW peatlands: prioritas pengelolaan air dan pencegahan kebakaran

**Pertanyaan bisnis.** Di provinsi mana paparan gambut terkonsentrasi, dan bagaimana anggaran pengelolaan air, sekat kanal, serta pencegahan berbasis masyarakat diprioritaskan? Notebook ini menggunakan layer gambut WRI/GFW dan batas administrasi, tanpa NASA FIRMS maupun data populasi.

> **Catatan temporal:** layer WRI/GFW ini tidak memiliki kolom tanggal atau tahun. Ia adalah baseline spasial statis yang dapat dioverlay untuk seluruh 2023–2026, bukan pengamatan gambut tahunan atau bukti perubahan gambut antar-tahun.

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'data').exists())
adm1 = gpd.read_file(ROOT / 'data/raw/boundaries/idn_adm1.geojson')
peat = gpd.read_file(ROOT / 'data/raw/peatland/kalimantan_peatlands.geojson')
name_col = next(c for c in ['shapeName', 'NAME_1', 'name'] if c in adm1.columns)
kalimantan = adm1[adm1[name_col].astype(str).str.contains('Kalimantan', case=False, na=False)][[name_col, 'geometry']].copy()
assert len(kalimantan) == 5, 'Batas ADM1 harus memuat lima provinsi Kalimantan'
outside_kalimantan = int((~peat.geometry.intersects(kalimantan.geometry.union_all())).sum())
temporal_columns = [c for c in peat.columns if any(word in c.lower() for word in ['date', 'year', 'time', 'period'])]
print(f'Validasi spasial: {len(peat):,} poligon; di luar Kalimantan: {outside_kalimantan}')
print(f'Kolom temporal sumber: {temporal_columns or "tidak ada — layer statis"}')
assert outside_kalimantan == 0, 'File kanonik harus sudah terpotong tepat ke Kalimantan'

equal_area = 'EPSG:6933'
provinces = kalimantan.to_crs(equal_area)
peat = peat.to_crs(equal_area)
intersection = gpd.overlay(provinces, peat[['geometry']], how='intersection')
peat_km2 = intersection.assign(peat_km2=intersection.area / 1e6).groupby(name_col)['peat_km2'].sum()
summary = provinces.assign(province_area_km2=provinces.area / 1e6).set_index(name_col)[['province_area_km2']].join(peat_km2).fillna(0)
summary['peat_share_pct'] = summary['peat_km2'] / summary['province_area_km2'] * 100
summary = summary.sort_values('peat_km2', ascending=False)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
summary['peat_km2'].plot(kind='barh', ax=axes[0], color='#6b4c9a')
axes[0].invert_yaxis()
axes[0].set(title='Cakupan gambut per provinsi', xlabel='Luas gambut (km²)', ylabel='')
provinces.to_crs('EPSG:4326').plot(ax=axes[1], color='#e8e8e8', edgecolor='white')
peat.to_crs('EPSG:4326').plot(ax=axes[1], color='#6b4c9a', alpha=.55, linewidth=.1)
axes[1].set_title('Jejak gambut dan wilayah operasi administrasi')
axes[1].set_axis_off()
plt.show()

largest = summary.iloc[0]
highest_share = summary['peat_share_pct'].idxmax()
print(f'Insight bisnis: {largest.name} memiliki jejak gambut absolut terbesar ({largest["peat_km2"]:.0f} km²); {highest_share} memiliki proporsi gambut tertinggi terhadap luas provinsi.')
print('Tindakan: gunakan luas absolut untuk ukuran kapasitas program, dan proporsi gambut untuk menentukan seberapa kuat SOP hidrologi perlu diintegrasikan ke perencanaan rutin provinsi.')

In [ ]:
# EDA tambahan: fragmentasi menentukan bentuk implementasi pengelolaan air.
fragmentation = intersection.assign(area_km2=intersection.area / 1e6).groupby(name_col).agg(
    polygon_count=('area_km2', 'size'), median_polygon_km2=('area_km2', 'median')
).sort_values('polygon_count')
fig, ax = plt.subplots(figsize=(9, 5))
for province, row in fragmentation.iterrows():
    ax.scatter(row['polygon_count'], row['median_polygon_km2'], s=90, color='#6b4c9a')
    ax.annotate(province, (row['polygon_count'], row['median_polygon_km2']), xytext=(5, 4), textcoords='offset points', fontsize=8)
ax.set(title='Fragmentasi gambut dan kompleksitas implementasi', xlabel='Jumlah fragmen poligon gambut', ylabel='Median luas fragmen (km²)')
ax.grid(alpha=.25)
plt.show()
print('Implikasi karhutla: banyak fragmen kecil berarti lebih banyak titik inspeksi, koordinasi lokal, dan pemantauan tersebar. Fragmen besar lebih cocok untuk pendekatan koridor hidrologi. Lengkapi dengan data kanal dan muka air sebelum menetapkan proyek.')

## Catatan keputusan

- Luas gambut adalah indikator paparan struktural, bukan bukti kebakaran atau kekeringan saat ini.
- Untuk pemodelan 2023–2026, gunakan layer ini sebagai fitur spasial tetap (*static covariate*); jangan membuat tren tahunan palsu dengan menyalin nilainya dan menganggapnya observasi baru.
- Gunakan luas absolut untuk ukuran program modal dan proporsi gambut untuk prioritas kebijakan serta model operasi.
- Overlay-kan dengan aset hidrologi, muka air, konsesi, dan batas masyarakat sebelum menentukan kerja lapangan.